# Lab #13 · PyTorch: Build a Complete Training Loop · **medium**

Реализуй `train_model(model, X_train, y_train, X_val, y_val, epochs, batch_size, lr)` — батчи, shuffle train, валидация, список dict с `epoch`, `train_loss`, `val_loss`, `val_accuracy`.

[https://www.deep-ml.com/labs/13](https://www.deep-ml.com/labs/13) · [README.md](./README.md) · эталон [lab_13_complete_training_loop_решения.ipynb](./lab_13_complete_training_loop_решения.ipynb) **после** попытки


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def need(cond, msg="допиши код"):
    assert cond, msg


def close(a, b, eps=1e-5):
    if a is None:
        return False
    return torch.allclose(torch.as_tensor(a, dtype=torch.float32),
                          torch.as_tensor(b, dtype=torch.float32), atol=eps)

import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


def train_model(model, X_train, y_train, X_val, y_val, epochs, batch_size, lr):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    loader = DataLoader(
        TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True
    )
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        n_batches = 0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            n_batches += 1
        train_loss /= max(n_batches, 1)

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val)
            val_loss = criterion(val_logits, y_val).item()
            val_acc = (val_logits.argmax(dim=1) == y_val).float().mean().item()

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
            }
        )
    return history

In [ ]:
model = nn.Linear(10, 2)
X_train = torch.randn(100, 10)
y_train = torch.randint(0, 2, (100,))
X_val = torch.randn(20, 10)
y_val = torch.randint(0, 2, (20,))
history = train_model(model, X_train, y_train, X_val, y_val, epochs=2, batch_size=16, lr=0.01)
need(isinstance(history, list) and len(history) == 2)
need(all("epoch" in h and "train_loss" in h and "val_loss" in h and "val_accuracy" in h for h in history))
print("все проверки пройдены")